# 02 — Classical Forecasting

LightGBM quantile forecaster (p10/p50/p90) vs. seasonal-naive baseline, on a real M5 series, evaluated with rolling-origin backtesting — never a random split. All metrics and mlflow records in this notebook are **LIVE**.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from qsci_core.data import loaders
from qsci_core.evaluation import metrics as metrics_mod
from qsci_core.evaluation.backtest import run_backtest
from qsci_core.forecasting.classical.lightgbm_forecaster import LightGBMQuantileForecaster, lightgbm_p50_point_forecast
from qsci_core.forecasting.classical.seasonal_naive import seasonal_naive_predict
from qsci_core import mlflow_utils

mlflow_utils.set_tracking_uri()
print("project root:", PROJECT_ROOT)


project root: D:\Projects\QML project


## Series selection

Status: **LIVE**. Selected by a disclosed, reproducible rule — the highest total-volume item in store `CA_1` — not hand-picked for a favorable-looking result.

In [2]:
m5 = loaders.load_m5(PROJECT_ROOT / "data" / "raw" / "m5")
sales, calendar = m5["sales"], m5["calendar"]

day_cols = [c for c in sales.columns if c.startswith("d_")]
sales_ca1 = sales[sales["store_id"] == "CA_1"].copy()
sales_ca1["total_sales"] = sales_ca1[day_cols].sum(axis=1)
top_row = sales_ca1.sort_values("total_sales", ascending=False).iloc[0]

series_id = top_row["id"]
print("selected series:", series_id, "| total historical units:", int(top_row["total_sales"]))

day_to_date = calendar.set_index("d")["date"]
series_vals = top_row[day_cols].astype(float)
series_df = pd.DataFrame({
    "date": pd.to_datetime(day_to_date.reindex(day_cols).values),
    "sales": series_vals.values,
}).sort_values("date").reset_index(drop=True)

print("series length (days):", len(series_df))
print("date range:", series_df["date"].min().date(), "to", series_df["date"].max().date())
print("zero-sales days:", (series_df["sales"] == 0).mean() * 100, "%")
series_df.tail()


selected series: FOODS_3_090_CA_1_evaluation | total historical units: 128855
series length (days): 1941
date range: 2011-01-29 to 2016-05-22
zero-sales days: 24.16280267903143 %


,date,sales
1936,2016-05-18,39.0
1937,2016-05-19,51.0
1938,2016-05-20,69.0
1939,2016-05-21,67.0
1940,2016-05-22,64.0


## Rolling-origin backtest — LightGBM (p50) vs. seasonal-naive

Status: **LIVE**. Never a random split — each fold trains only on data strictly before its 14-day test window, walking forward through real history. FVA is measured directly, not asserted.

In [3]:
HORIZON = 14
N_SPLITS = 5
MIN_TRAIN_SIZE = 400
SEED = 42
np.random.seed(SEED)

def lgb_fit_predict(train_df, test_df):
    return lightgbm_p50_point_forecast(train_df, test_df, "date", "sales")

def naive_fit_predict(train_df, test_df):
    return seasonal_naive_predict(train_df, test_df, "sales", season_length=7)

lgb_result = run_backtest(series_df, "date", "sales", lgb_fit_predict, HORIZON, N_SPLITS, MIN_TRAIN_SIZE, baseline_fn=naive_fit_predict)
naive_result = run_backtest(series_df, "date", "sales", naive_fit_predict, HORIZON, N_SPLITS, MIN_TRAIN_SIZE)

print("=== LightGBM (p50) — per fold ===")
print(lgb_result.to_dataframe()[["fold_id", "test_start", "test_end", "rmse", "mae", "wmape", "bias", "fva"]])
print("\nLightGBM mean metrics:", lgb_result.mean_metrics)

print("\n=== Seasonal-naive — per fold ===")
print(naive_result.to_dataframe()[["fold_id", "test_start", "test_end", "rmse", "mae", "wmape", "bias"]])
print("\nSeasonal-naive mean metrics:", naive_result.mean_metrics)


=== LightGBM (p50) — per fold ===
   fold_id test_start   test_end       rmse        mae      wmape       bias  \
0        0 2016-03-14 2016-03-27  16.501926  10.667506  19.073446  -5.893560   
1        1 2016-03-28 2016-04-10  22.007091  18.494229  38.587065  17.728164   
2        2 2016-04-11 2016-04-24  13.900809  11.241867  20.929007   3.206221   
3        3 2016-04-25 2016-05-08  12.913376  10.834205  16.834503  -6.096649   
4        4 2016-05-09 2016-05-22  18.075079  13.711111  25.559994  10.369596   

        fva  
0  0.486787  
1  0.078579  
2  0.197010  
3  0.331811  
4  0.311987  

LightGBM mean metrics: {'rmse': 16.679656296916104, 'mae': 12.989783625046226, 'wmape': 24.196803167921814, 'bias': 3.8627545118364104, 'fva': 0.2812347798263138}

=== Seasonal-naive — per fold ===
   fold_id test_start   test_end       rmse        mae      wmape       bias
0        0 2016-03-14 2016-03-27  28.614432  20.785714  37.164751  -3.785714
1        1 2016-03-28 2016-04-10  25.058503  20.

## mlflow tracking

Status: **LIVE**. Every run logs dataset version, seed, params, metrics, runtime, and git commit — per CLAUDE.md §7.

In [4]:
import mlflow

with mlflow_utils.tracked_run(
    "m5_classical_forecasting", f"lightgbm_{series_id}", dataset_version="m5-forecasting-accuracy-evaluation",
    seed=SEED, params={"model": "lightgbm_quantile_p50", "horizon": HORIZON, "n_splits": N_SPLITS, "min_train_size": MIN_TRAIN_SIZE, "series_id": series_id},
) as log:
    log(lgb_result.mean_metrics)

with mlflow_utils.tracked_run(
    "m5_classical_forecasting", f"seasonal_naive_{series_id}", dataset_version="m5-forecasting-accuracy-evaluation",
    seed=SEED, params={"model": "seasonal_naive", "season_length": 7, "horizon": HORIZON, "n_splits": N_SPLITS, "min_train_size": MIN_TRAIN_SIZE, "series_id": series_id},
) as log:
    log(naive_result.mean_metrics)

runs = mlflow.search_runs(experiment_names=["m5_classical_forecasting"])
print(runs[["tags.mlflow.runName", "params.seed", "tags.git_commit", "metrics.wmape", "metrics.runtime_seconds"]])


                          tags.mlflow.runName params.seed  \
0  seasonal_naive_FOODS_3_090_CA_1_evaluation          42   
1        lightgbm_FOODS_3_090_CA_1_evaluation          42   
2  seasonal_naive_FOODS_3_090_CA_1_evaluation          42   
3        lightgbm_FOODS_3_090_CA_1_evaluation          42   

                            tags.git_commit  metrics.wmape  \
0  0fbeac6a10fffdad02892249ff0f0fca06636dfd      33.490214   
1  0fbeac6a10fffdad02892249ff0f0fca06636dfd      24.196803   
2  0fbeac6a10fffdad02892249ff0f0fca06636dfd      33.490214   
3  0fbeac6a10fffdad02892249ff0f0fca06636dfd      24.196803   

   metrics.runtime_seconds  
0                 0.100196  
1                 0.361253  
2                 0.102270  
3                 0.423084  


## Forward quantile forecast (p10/p50/p90)

Status: **LIVE**. Fit on the full available history, forecasting 14 days beyond the last real observed date — quantile-crossing rectified (see `lightgbm_forecaster.py`).

In [5]:
forecaster = LightGBMQuantileForecaster().fit(series_df, "date", "sales")
future_dates = pd.date_range(series_df["date"].max() + pd.Timedelta(days=1), periods=HORIZON, freq="D")
quantile_preds = forecaster.predict(series_df, horizon=HORIZON, future_dates=future_dates)

forecast_df = pd.DataFrame({
    "date": future_dates,
    "p10": quantile_preds[0.1],
    "p50": quantile_preds[0.5],
    "p90": quantile_preds[0.9],
})
print(forecast_df)

# day-1-ahead forecast is what feeds the recommendation below
next_p10, next_p50, next_p90 = forecast_df.iloc[0][["p10", "p50", "p90"]]
# test_df's content is unused beyond its length (=1) by seasonal_naive_predict; this gives a single real next-day naive reference
naive_next = seasonal_naive_predict(series_df, forecast_df.iloc[:1], "sales", season_length=7)[0]
print(f"\nnext-day: p10={next_p10:.1f} p50={next_p50:.1f} p90={next_p90:.1f} | seasonal-naive reference={naive_next:.1f}")

         date        p10        p50         p90
0  2016-05-23  35.924778  43.230049   64.468218
1  2016-05-24  29.294888  38.432013   57.550610
2  2016-05-25  29.317081  40.115750   53.792986
3  2016-05-26  32.177464  53.144010   65.599286
4  2016-05-27  52.664292  79.046185   99.711844
5  2016-05-28  56.268601  82.405204   99.127437
6  2016-05-29  56.964592  70.507626   98.499616
7  2016-05-30  33.312150  44.648013   73.087757
8  2016-05-31  28.862181  42.324905   51.345321
9  2016-06-01  30.471092  42.053984   52.772449
10 2016-06-02  33.596640  50.294498   62.339246
11 2016-06-03  53.169972  74.837318  100.277616
12 2016-06-04  58.897103  83.106970  105.265680
13 2016-06-05  56.639617  72.718925  100.496803

next-day: p10=35.9 p50=43.2 p90=64.5 | seasonal-naive reference=38.0


C:\Users\piush\AppData\Local\Temp\ipykernel_3984\1475824934.py:2: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  future_dates = pd.date_range(series_df["date"].max() + pd.Timedelta(days=1), periods=HORIZON, freq="D")


## Confidence decomposition

Status: **LIVE**. Each component computed per the documented formula in `qsci_core/confidence/decomposition.py` — no invented percentage.

In [6]:
from qsci_core.data.quality import assess_dataframe
from qsci_core.confidence.decomposition import (
    backtest_performance_score, compute_confidence, data_quality_score,
    forecast_stability_score, model_consensus_score,
)

quality_report = assess_dataframe(series_df, dataset_name=series_id, numeric_cols=["sales"])
missing_rate = quality_report.missing_pct.get("sales", 0.0) / 100
negative_rate = quality_report.negative_value_counts.get("sales", 0) / len(series_df)
duplicate_rate = quality_report.duplicate_rows / len(series_df)

dq_score = data_quality_score(missing_rate, negative_rate, duplicate_rate)
fs_score = forecast_stability_score(next_p10, next_p50, next_p90)
mc_score = model_consensus_score(next_p50, naive_next)
bp_score = backtest_performance_score(lgb_result.mean_metrics["wmape"])

confidence = compute_confidence(dq_score, fs_score, mc_score, bp_score)
print(confidence.to_dict())


{'data_quality': 1.0, 'forecast_stability': np.float64(0.6023122097616477), 'model_consensus': 0.8790182117963193, 'backtest_performance': 0.7580319683207819, 'overall': np.float64(0.8098405974696872), 'weights': {'data_quality': 0.25, 'forecast_stability': 0.25, 'model_consensus': 0.25, 'backtest_performance': 0.25}, 'status': 'LIVE'}


## Risk, financial impact, and recommendation

Status: **LIVE** math over **ILLUSTRATIVE** cost inputs — M5 has no real per-unit holding/ordering/stockout cost data, so `CostAssumptions` below is a documented assumption, not a fact. The formulas themselves (EOQ, safety stock, reorder point, stockout risk) run for real.

In [7]:
from qsci_core.financial.eoq import CostAssumptions, economic_order_quantity, reorder_point as calc_reorder_point, safety_stock, total_inventory_cost
from qsci_core.risk.stockout_risk import demand_volatility, stockout_risk_from_quantiles
from qsci_core.recommendation.builder import build_recommendation

# Lead time is fixed at 1 day so the reorder point and the forecast quantiles
# are on the same (single-day) basis. A real multi-day lead time would need
# the *cumulative* demand distribution over that window, not a naive
# quantile*days scaling (that's statistically wrong for quantiles, unlike
# the mean) -- out of scope for this Phase 1 illustrative demo.
LEAD_TIME_DAYS = 1
SERVICE_Z_95 = 1.645

costs = CostAssumptions(holding_cost_per_unit_per_year=1.5, ordering_cost_per_order=35.0, stockout_cost_per_unit=8.0)
recent_history = series_df["sales"].to_numpy()[-90:]
demand_std = recent_history.std(ddof=1)
avg_daily_demand = recent_history.mean()  # trailing-90-day average, not a single volatile day's forecast
cv = demand_volatility(recent_history)
print(f"trailing-90-day avg daily demand: {avg_daily_demand:.2f}, std: {demand_std:.2f}, coefficient of variation: {cv:.2f}")

eoq = economic_order_quantity(annual_demand=avg_daily_demand * 365, costs=costs)
ss = safety_stock(demand_std=demand_std, lead_time_days=LEAD_TIME_DAYS, service_z=SERVICE_Z_95)
rp = calc_reorder_point(avg_daily_demand=avg_daily_demand, lead_time_days=LEAD_TIME_DAYS, safety_stock_units=ss)
fin = total_inventory_cost(avg_inventory_units=eoq / 2, orders_per_year=365 * avg_daily_demand / eoq, expected_stockout_units=cv * avg_daily_demand, costs=costs)

risk = stockout_risk_from_quantiles(next_p10, next_p50, next_p90, reorder_point=rp)

recommendation = build_recommendation(
    sku_id=series_id, forecast_p10=next_p10, forecast_p50=next_p50, forecast_p90=next_p90,
    suggested_order_qty=eoq, reorder_point=rp, risk=risk, confidence=confidence, financial_impact=fin,
)
print(f"reorder point (1-day basis): {rp:.1f} vs. next-day forecast p10={next_p10:.1f} p50={next_p50:.1f} p90={next_p90:.1f}")
print(recommendation.rationale)

trailing-90-day avg daily demand: 54.49, std: 21.60, coefficient of variation: 0.40
reorder point (1-day basis): 90.0 vs. next-day forecast p10=35.9 p50=43.2 p90=64.5
Median demand forecast is 43.2 units (p10=35.9, p90=64.5). Suggested order quantity is 963.4 units to cover the reorder point of 90.0 units. Estimated stockout risk is 0.0% (LOW), computed via normal_approx_from_quantile_spread. Overall confidence is 81.0%, decomposed as data quality=1.00, forecast stability=0.60, model consensus=0.88, backtest performance=0.76. Financial impact figures are ILLUSTRATIVE — based on documented cost assumptions, not real per-unit cost data.


## Persistence: Recommendation → Approval → Audit

Status: **LIVE**. Written to a real SQLite database (`experiments/qsci_dev.db`). Outcome recording is intentionally skipped here — the forecasted period hasn't happened yet, so there's no real actual-demand to record; fabricating one would violate the no-feature-theater rule. `qsci_core.persistence.db.record_outcome` exists and is tested (`tests/test_recommendation_persistence.py`) for when real outcome data becomes available.

In [8]:
from qsci_core.persistence import db
from qsci_core.persistence.models import ApprovalDecision, AuditRecord

engine = db.get_engine()
session = db.get_session(engine)

rec_row = db.persist_recommendation(session, recommendation)
approval_row = db.record_approval(session, rec_row.id, ApprovalDecision.APPROVE, approver="notebook_demo_planner", notes="Phase 1 golden-path demonstration")

events = session.query(AuditRecord).filter_by(recommendation_id=rec_row.id).order_by(AuditRecord.id).all()
print("persisted recommendation id:", rec_row.id)
print("approval:", approval_row.decision, "by", approval_row.approver)
print("audit trail:", [e.event_type for e in events])
session.close()
engine.dispose()


persisted recommendation id: 2
approval: ApprovalDecision.APPROVE by notebook_demo_planner
audit trail: ['RECOMMENDATION_CREATED', 'APPROVAL_APPROVE']


## Honest interpretation

Reported as measured, not adjusted to look better: whichever model actually wins on this series' backtest is stated as such below (see printed FVA above). This is a single real series, not a claim about M5-wide performance — RQ1/H1 at scale is Phase 2's job, run against the *same* classical baseline established here.

In [9]:
fva = lgb_result.mean_metrics["fva"]
if fva > 0:
    print(f"LightGBM beat seasonal-naive on this series: FVA = {fva:.1%} (positive = improvement).")
else:
    print(f"LightGBM did NOT beat seasonal-naive on this series: FVA = {fva:.1%} (negative/zero = no improvement or worse). Reporting as-is, per CLAUDE.md §1.")

print(f"LightGBM wMAPE: {lgb_result.mean_metrics['wmape']:.2f}%  |  Seasonal-naive wMAPE: {naive_result.mean_metrics['wmape']:.2f}%")


LightGBM beat seasonal-naive on this series: FVA = 28.1% (positive = improvement).
LightGBM wMAPE: 24.20%  |  Seasonal-naive wMAPE: 33.49%
